# Looped transformer (`variant: looped`, `arch: looped`)

<img src="../../assets/explainers/looped.gif" width="100%">

**The loop**: one block of weights $W$ applied again and again - a state dot laps the block while the unrolled strip fills; compute depth grows, parameters do not. **One forward, slowed down**: after every pass the ring under each character fills with its ACT halting probability until it stops, and the logit lens shows what the model would predict if it stopped there. **Generation**: one character per forward pass, with a bar under each showing how many loop passes its position used.

## Intuition

A standard transformer has $L$ distinct layers. A looped transformer has **one** block and runs it $T$
times, refining a hidden state - depth becomes iteration count (Universal Transformers, Dehghani et al.
2019). Because the same weights run at every pass, the model must learn a good *iterative operator*;
in exchange it can think longer at test time, or let each token decide how long to think (ACT).

## Math

$$e = \mathrm{prelude}(\mathrm{embed}(x)),\qquad h_0 = 0,\qquad
  h_t = f_W\big(h_{t-1} + e\big) + E^t \;\; (t = 1..T),\qquad \hat y = \mathrm{head}(\mathrm{coda}(h_T)).$$

* **Input injection** ($+e$ every pass) keeps the input visible however long the loop runs.
* **Depth embedding** $E^t$ tells the shared block which pass it is in (sinusoidal: defined for any $t$).
* **ACT** (Graves 2016): per token $p^i_t = \sigma(w\cdot h^i_t + b)$; the token halts when
  $\sum_t p^i_t$ would exceed $1-\epsilon$, its output is $\sum_t \lambda^i_t h^i_t$ (λ = $p$, then the
  remainder $R^i$), and the loss adds the **ponder cost** $\tau \cdot \overline{N^i + R^i}$.
* **Gradients.** Backprop through $T$ tied passes multiplies Jacobians $\prod_t (I + J_{f_W})$; the
  residual form, gradient clipping, optional sandwich norms and truncated backprop (`bptt_loops`) keep it
  stable. Running to a fixed point $h^* = f_W(h^*)$ is the DEQ view (`exit_tolerance`).

## In the code

| File | What |
|---|---|
| HaleBlocks `hale_core/nn/stacks/looped.py` | `LoopSpec`, `LoopedStack`, `loop_budget`, `loop_stats`, `loop_aux_loss` |
| HaleBlocks `hale_core/nn/layers/loop.py` | `LoopDepthEmbedding`, `LoopInputInjection`, `ACTHalting` |
| `models_looped/model.py` | `LoopedLM` (`forward_with_loops`, `last_loop_stats`) |
| `models_looped/{loss,sample,metrics}.py` | NLL (ponder added by the trainer), loop-budget sampler, `loop_scaling`, `ponder_histogram` |
| `losses/auxiliary.py` | `with_auxiliary_losses`: task loss + ponder cost, used by the trainer |

In [ ]:
import sys
from pathlib import Path

# Run from anywhere inside the repo: put src/ on the path if hale_llm is not installed.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from IPython.display import HTML, display

import hale_llm.models  # noqa: F401  - registers every paradigm
from hale_llm.core.registry import get_loss, get_sampler, get_variant
from hale_llm.dataloader.collate import get_collate
from hale_llm.visualization.explainers import CORPUS, get_explainer

torch.manual_seed(0)
plt.rcParams["figure.dpi"] = 110
print("training corpus:")
for sentence in CORPUS:
    print("  ", repr(sentence))

## Train a toy model

One shared layer, up to 6 passes, ACT halting with ponder weight τ = 0.003. The explainer's recipe trains a ~100k-parameter model on the four corpus sentences
in a few seconds on a CPU - small enough to inspect every number, trained enough to show real
behaviour.

In [ ]:
explainer = get_explainer("looped")
run = explainer.train()
tok, model, cfg = run.tokenizer, run.model, run.cfg
print(type(model).__name__, f"{sum(p.numel() for p in model.parameters()):,} parameters")
print("config:", {k: v for k, v in cfg.model.model_dump().items()
                  if k in ("arch", "d_model", "n_layers", "attn_type", "use_time_cond",
                           "block_size", "n_mtp_heads", "fm_schedule")})

fig, ax = plt.subplots(figsize=(6.5, 2.4))
ax.semilogy(run.losses, color="#264653")
ax.set(xlabel="step", ylabel="training loss", title="toy training")
plt.show()

## Parameters vs compute

In [ ]:
stack = model.backbone.stack
core = sum(p.numel() for p in stack.core.parameters())
print(stack.spec)
print(f"shared block: {core:,} params | effective depth {stack.effective_depth} layers "
      f"| a stacked model of that depth would need {stack.effective_depth * core:,}")

## One forward in slow motion: logit lens and halting

In [ ]:
trace = explainer.trace(run, seed=0)
for t, p in enumerate(trace.passes):
    guess = ", ".join(f"{tok.token(i)!r}:{q:.2f}" for i, q in zip(p.lens_ids[:3], p.lens_probs[:3]))
    halted = sum(not r for r in p.running)
    print(f"pass {t + 1}: next char [{guess}]   halted {halted}/8   halting mass "
          + " ".join(f"{h:.2f}" for h in p.halting))

## Passes per token during generation

In [ ]:
n = [s.updates for s in trace.steps]
fig, ax = plt.subplots(figsize=(9, 2.6))
ax.bar(range(len(n)), n, color="#F4A261")
ax.set_xticks(range(len(n)), list(tok.decode(trace.ids[0])[8:].replace(" ", "␣")), fontsize=8)
ax.set(ylabel="passes N(i)", title=f"ACT passes per generated character (max {stack.spec.n_loops})")
plt.show()

## Test-time compute: loss vs number of loops

`forward_with_loops(batch, n)` runs a different loop budget at inference. ACT caps the passes per token,
so extra budget only helps tokens that have not halted; with `halting: none` and `min_loops` training
(`configs/looped/recurrent_depth.yaml`) the loss curve keeps improving past the training $T$.

In [ ]:
batch = get_collate("looped")(torch.tensor([tok.encode(s) for s in CORPUS]), tok)
loss_fn = get_loss("looped")
budgets = [1, 2, 3, 4, 6, 8, 12]
with torch.no_grad():
    losses = []
    for b in budgets:
        from hale_core.nn.stacks import loop_budget
        with loop_budget(model, b):
            losses.append(float(loss_fn(model, batch)))
fig, ax = plt.subplots(figsize=(5, 2.6))
ax.plot(budgets, losses, "o-", color="#264653")
ax.axvline(stack.spec.n_loops, color="#E76F51", ls="--", label="training T")
ax.set(xlabel="loop budget", ylabel="NLL", yscale="log", title="loss vs passes"); ax.legend()
plt.show()

In [ ]:
# Re-render the animation from this notebook's model. The committed copy shown at the top is
# assets/explainers/looped.gif (regenerate all of them with `hale-llm-explain`).
from PIL import Image

out = ROOT / "notebooks" / "models" / "outputs" / "looped.gif"  # git-ignored
explainer.build(out, run=run)
with Image.open(out) as gif:
    print(out.relative_to(ROOT), f"{gif.n_frames} frames, {gif.size[0]}x{gif.size[1]},",
          f"{out.stat().st_size / 1024:.0f} KB")
# In Jupyter: display(HTML('<img src="outputs/looped.gif" width="100%">'))

## Takeaways

* Parameters of one block, depth of many; the loop count is a knob at inference.
* ACT makes depth per token adaptive; τ trades compute for loss (raise it and watch the histogram shift).
* Training is BPTT over shared weights - clip gradients; use `bptt_loops` for memory, `sandwich_norm` for
  long loops. Passes are sequential, so latency grows with $T$.

**Exercises.** (1) Retrain with `"ponder_weight": 0.05` and compare passes per token. (2) Train
`halting: none` with `min_loops: 2` and plot the loss vs loop budget again. (3) Add `scratch_tokens: 4`.